# KNN Rice Classification — Data Mining Project

This notebook follows the assignment requirements and implements **K-Nearest Neighbors from scratch** using:
- data inspection (EDA),
- missing-value and duplicate handling,
- manual standardization,
- Euclidean distance,
- majority voting.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


## 1. Importing the dataset


In [ ]:
DATA_PATH = Path("../data/Rice_Cammeo_Osmancik.csv")
df = pd.read_csv(DATA_PATH)
df.head()


## 2. EDA


In [ ]:
print("Column types:\n", df.dtypes)
print("\nFirst 10 rows:\n", df.head(10))
print("\nLast 10 rows:\n", df.tail(10))
print("\nShape:", df.shape)

numeric_preview = df.replace("???", np.nan).drop(columns=["Class"], errors="ignore")
numeric_preview = numeric_preview.apply(pd.to_numeric, errors="coerce")
print("\nMinimum values:\n", numeric_preview.min())
print("\nMaximum values:\n", numeric_preview.max())


## 3. Data cleaning


In [ ]:
df = df.replace("???", np.nan)

print("Missing values before cleaning:\n", df.isna().sum())
print("\nDuplicate rows before cleaning:", df.duplicated().sum())

df = df.dropna().drop_duplicates().reset_index(drop=True)

feature_columns = [c for c in df.columns if c != "Class"]
for col in feature_columns:
    df[col] = pd.to_numeric(df[col], errors="raise")

print("\nMissing values after cleaning:\n", df.isna().sum())
print("\nDuplicate rows after cleaning:", df.duplicated().sum())
print("\nData types after cleaning:\n", df.dtypes)


## 4. Data normalization


In [ ]:
means = df[feature_columns].mean()
stds = df[feature_columns].std().replace(0, 1)

normalized_df = df.copy()
normalized_df[feature_columns] = (df[feature_columns] - means) / stds

print("Means after normalization:\n", normalized_df[feature_columns].mean())
print("\nStandard deviations after normalization:\n", normalized_df[feature_columns].std())


## 5. KNN from scratch


In [ ]:
def normalize_new_sample(sample_values, means, stds):
    sample = pd.Series(sample_values, index=means.index, dtype=float)
    return ((sample - means) / stds).to_numpy(dtype=float)

def knn_predict(normalized_df, normalized_sample, k=61):
    X = normalized_df[feature_columns].to_numpy(dtype=float)
    y = normalized_df["Class"].to_numpy()

    distances = np.linalg.norm(X - normalized_sample, axis=1)
    nearest_indices = np.argsort(distances)[:k]
    nearest_labels = pd.Series(y[nearest_indices])

    return nearest_labels.mode().iloc[0]


## 6. Get a new sample and predict


In [ ]:
new_sample = {}
for feature in feature_columns:
    new_sample[feature] = float(input(f"Enter {feature}: "))

normalized_sample = normalize_new_sample(new_sample, means, stds)

k = 61
prediction = knn_predict(normalized_df, normalized_sample, k=k)
print(f"Predicted rice class (k={k}): {prediction}")
